# Chirundu Town Council — 2025 Bi-Annual Performance Report Extraction
## CSC 4792: Data Mining and Warehousing — Mini Project

---

## 1. Introduction

This notebook extracts financial data from the **Chirundu Town Council 
2025 Bi-Annual Performance Report** covering January–June 2025.

### Source
- **File:** `2025 BI ANNUAL PERFORMANCE REPORT.pdf` (13 pages)
- **Coverage:** 1 January 2025 – 30 June 2025 (H1)
- **Status:** Unaudited mid-year position

### ⚠️ Structural Difference vs Prior Years
Unlike the 2022–2024 Financial Statements (full-year audited), this 
document:
- Covers **6 months only** (H1)
- Contains **no monthly LGEF breakdown**
- Contains **no detailed CDF project spending**
- Uses **full-year budget** vs **half-year actual** comparison

### Assigned Scope — Finances
| # | Dimension | Source |
|---|---|---|
| 1 | Approved Budgets (vs H1 Actual) | p.1 |
| 2 | LGEF Usage (H1 aggregate) | p.1 |
| 3 | Local Revenue (H1 aggregate) | p.1 |
| 4 | Internal Audit Findings | p.2 |
| 5 | Debt & Arrears | p.3 |

### Deliverable
`db-unza26-csc4792-chirundu_2025_financials.csv`

### Known Data Quality Notes
- **Pages 4, 6, 10** contain scan artefacts ("1 1 1 1...") — excluded.
- **Pages 5, 7–9, 11–13** contain output indicator data (performance 
  targets, not financial). These are outside the finances scope — 
  another group member may extract them separately.

In [1]:
# ============================================================
# SETUP — 2025 Bi-Annual
# ============================================================
from pathlib import Path
import re
import pandas as pd
import pymupdf
from PIL import Image, ImageOps
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Users\USER\Desktop\tesseract.exe"

PDF_PATH = Path(r"C:\Users\USER\Downloads\2025 BI ANNUAL PERFORMANCE REPORT.pdf")
OUTPUT_DIR = Path("finances_output")
OUTPUT_DIR.mkdir(exist_ok=True)

YEAR = 2025
PERIOD = "H1"

assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"
print(f"✅ PDF: {PDF_PATH.name}")
print(f"✅ Tesseract: {pytesseract.get_tesseract_version()}")

✅ PDF: 2025 BI ANNUAL PERFORMANCE REPORT.pdf
✅ Tesseract: 5.5.3.20260724


## 2. OCR Pipeline

Same unified pipeline as 2022–2024: PyMuPDF at 300 DPI, grayscale + 
binarise at 180, Tesseract with adaptive PSM.

### PSM Choice
| Page | Content | PSM |
|---|---|---|
| 1 | Financial tables | 4 |
| 2 | Audit findings table | 11 |
| 3 | Debt & arrears table | 11 |
| 5–13 | Output indicators | 6 |

In [2]:
def ocr_page(pdf_path, page_number, psm=4):
    doc = pymupdf.open(pdf_path)
    try:
        page = doc[page_number - 1]
        pix = page.get_pixmap(matrix=pymupdf.Matrix(300/72, 300/72), alpha=False)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        img = ImageOps.grayscale(img)
        img = img.point(lambda px: 0 if px < 180 else 255)
        return pytesseract.image_to_string(img, config=f"--oem 3 --psm {psm}")
    finally:
        doc.close()


def psm_for_page(n):
    if n == 1: return 4
    if n in (2, 3): return 11
    if n >= 5: return 6
    return 4


doc = pymupdf.open(PDF_PATH)
total_pages = len(doc)
doc.close()

rows = []
for page_num in range(1, total_pages + 1):
    psm = psm_for_page(page_num)
    print(f"OCR page {page_num}/{total_pages} (PSM {psm})...")
    rows.append({
        "year": YEAR, "period": PERIOD, "page": page_num, "psm": psm,
        "ocr_text": ocr_page(PDF_PATH, page_num, psm=psm),
        "source_file": PDF_PATH.name,
    })

raw_ocr = pd.DataFrame(rows)
raw_ocr.to_csv(OUTPUT_DIR / f"chirundu_{YEAR}_raw_ocr.csv",
               sep="|", index=False, encoding="utf-8-sig")
print(f"\n✅ OCR: {len(raw_ocr)} pages")


def is_garbled(text):
    return bool(re.search(r"(?:\b1\s+){40,}", text or ""))

raw_ocr["is_garbled"] = raw_ocr["ocr_text"].apply(is_garbled)
garbled = raw_ocr.loc[raw_ocr["is_garbled"], "page"].tolist()
print(f"⚠️  Garbled pages: {garbled}")

OCR page 1/13 (PSM 4)...
OCR page 2/13 (PSM 11)...
OCR page 3/13 (PSM 11)...
OCR page 4/13 (PSM 4)...
OCR page 5/13 (PSM 6)...
OCR page 6/13 (PSM 6)...
OCR page 7/13 (PSM 6)...
OCR page 8/13 (PSM 6)...
OCR page 9/13 (PSM 6)...
OCR page 10/13 (PSM 6)...
OCR page 11/13 (PSM 6)...
OCR page 12/13 (PSM 6)...
OCR page 13/13 (PSM 6)...

✅ OCR: 13 pages
⚠️  Garbled pages: []


## 3. Receipts — H1 2025 (Page 1)

Extracted verbatim from the Receipts table on page 1. Column structure:

| Column | Meaning |
|---|---|
| `approved_budget_zmw` | Full-year 2025 budget |
| `actual_zmw` | Actual receipts as at 30 June 2025 |
| `variance_zmw` | Budget − Actual |
| `variance_pct` | Actual / Budget × 100 |

All values are in **Zambian Kwacha (ZMW)**.

In [3]:
# ============================================================
# RECEIPTS — H1 2025 (verbatim from document)
# ============================================================
RECEIPTS_H1 = [
    # (category, line_item, approved_budget, actual_h1)
    ("National Support", "Local Government Equalisation Fund",
     8399999.98, 3056652.61),
    ("National Support", "Grants In Lieu of Rates",
     150000.00, 150000.00),
    ("National Support", "Sector Grants",
     13205709.00, 4697433.69),
    ("National Support", "ZDSP Capital Grants",
     8340000.00, 0.00),
    ("National Support", "Constituency Development Fund",
     36058150.60, 20287381.52),
    ("National Support", "Other Grants",
     3200587.00, 0.00),

    ("Own Source Revenue", "Local Taxes", 795014.43, 274579.26),
    ("Own Source Revenue", "Fee & Charges", 29171743.17, 11372766.20),
    ("Own Source Revenue", "Licences", 505480.00, 141606.00),
    ("Own Source Revenue", "Levies", 231386.67, 63477.50),
    ("Own Source Revenue", "Permits", 2561580.00, 1244991.20),
    ("Own Source Revenue", "Commercial ventures", 1000000.00, 0.00),
    ("Own Source Revenue", "Others OSR", 309765.00, 355200.00),

    ("Other revenue", "Other revenue", 34574969.27, 13452620.26),
]

receipts_df = pd.DataFrame([
    {"fiscal_year": YEAR, "period": PERIOD,
     "record_type": "receipt",
     "section": cat,
     "line_item": item,
     "approved_budget_zmw": bud,
     "actual_zmw": act,
     "variance_zmw": round(bud - act, 2),
     "variance_pct": round((act / bud) * 100, 1) if bud else None,
     "source_document": PDF_PATH.name,
     "source_page": 1}
    for cat, item, bud, act in RECEIPTS_H1
])

print(f"✅ Receipts H1: {len(receipts_df)} rows")
print(f"   Budget total : K{receipts_df['approved_budget_zmw'].sum():,.2f}")
print(f"   Actual H1    : K{receipts_df['actual_zmw'].sum():,.2f}")
receipts_df

✅ Receipts H1: 14 rows
   Budget total : K138,504,385.12
   Actual H1    : K55,096,708.24


,fiscal_year,period,record_type,section,line_item,approved_budget_zmw,actual_zmw,variance_zmw,variance_pct,source_document,source_page
0,2025,H1,receipt,National Support,Local Government Equalisation Fund,8399999.98,3056652.61,5343347.37,36.4,2025 BI ANNUAL PERFORMANCE REPORT.pdf,1
1,2025,H1,receipt,National Support,Grants In Lieu of Rates,150000.00,150000.00,0.00,100.0,2025 BI ANNUAL PERFORMANCE REPORT.pdf,1
2,2025,H1,receipt,National Support,Sector Grants,13205709.00,4697433.69,8508275.31,35.6,2025 BI ANNUAL PERFORMANCE REPORT.pdf,1
3,2025,H1,receipt,National Support,ZDSP Capital Grants,8340000.00,0.00,8340000.00,0.0,2025 BI ANNUAL PERFORMANCE REPORT.pdf,1
4,2025,H1,receipt,National Support,Constituency Development Fund,36058150.60,20287381.52,15770769.08,56.3,2025 BI ANNUAL PERFORMANCE REPORT.pdf,1
5,2025,H1,receipt,National Support,Other Grants,3200587.00,0.00,3200587.00,0.0,2025 BI ANNUAL PERFORMANCE REPORT.pdf,1
6,2025,H1,receipt,Own Source Revenue,Local Taxes,795014.43,274579.26,520435.17,34.5,2025 BI ANNUAL PERFORMANCE REPORT.pdf,1
7,2025,H1,receipt,Own Source Revenue,Fee & Charges,29171743.17,11372766.20,17798976.97,39.0,2025 BI ANNUAL PERFORMANCE REPORT.pdf,1
8,2025,H1,receipt,Own Source Revenue,Licences,505480.00,141606.00,363874.00,28.0,2025 BI ANNUAL PERFORMANCE REPORT.pdf,1
9,2025,H1,receipt,Own Source Revenue,Levies,231386.67,63477.50,167909.17,27.4,2025 BI ANNUAL PERFORMANCE REPORT.pdf,1


## 4. Payments — H1 2025 (Page 1)

Extracted verbatim from the Payments table. The document states 
**Total payments** of K103,929,415.85 (budget) vs K43,832,931.75 (actual).

### Note
The document reports a **Net Budget Performance** of K(2,188,843.67) — a 
mid-year deficit. This is because H1 payments slightly exceed H1 receipts.

In [4]:
# ============================================================
# PAYMENTS — H1 2025 (verbatim from document)
# ============================================================
PAYMENTS_H1 = [
    ("Personal emoluments", 21313109.10, 10505097.94),
    ("Use of goods and services", 27592550.57, 13771587.68),
    ("Financial charges", 0, 0),
    ("Social benefits", 18879509.22, 9600022.19),
    ("Non-financial assets", 32426979.43, 5564972.94),
    ("Financial assets", 3717267.52, 4391251.00),
    ("Loan repayments", 0, 0),
    ("Other repayments", 0, 0),
]

payments_df = pd.DataFrame([
    {"fiscal_year": YEAR, "period": PERIOD,
     "record_type": "payment",
     "section": "",
     "line_item": item,
     "approved_budget_zmw": bud,
     "actual_zmw": act,
     "variance_zmw": round(bud - act, 2),
     "variance_pct": round((act / bud) * 100, 1) if bud else None,
     "source_document": PDF_PATH.name,
     "source_page": 1}
    for item, bud, act in PAYMENTS_H1
])

print(f"✅ Payments H1: {len(payments_df)} rows")
print(f"   Budget total : K{payments_df['approved_budget_zmw'].sum():,.2f}")
print(f"   Actual H1    : K{payments_df['actual_zmw'].sum():,.2f}")

# Validate against document totals
_bud_total = 103_929_415.85
_act_total = 43_832_931.75
print(f"\n   Document budget total: K{_bud_total:,.2f}")
print(f"   Document actual total: K{_act_total:,.2f}")

✅ Payments H1: 8 rows
   Budget total : K103,929,415.84
   Actual H1    : K43,832,931.75

   Document budget total: K103,929,415.85
   Document actual total: K43,832,931.75


## 5. Internal Audit Findings (Page 2)

Eight audit findings from the Semi-Annual Internal Audit covering:

1. Governance (committees & meetings)
2. Budget Execution
3. Income — Own Source Revenue
4. Accounting procedures and controls
5. Stores management
6. Procurement procedures and controls
7. Payroll management
8. Risk management

Each row captures the finding, recommendation, and management action.

In [5]:
# ============================================================
# AUDIT FINDINGS — H1 2025 (verbatim from page 2)
# ============================================================
AUDIT_FINDINGS = [
    (1, "Governance (Established Committees & Meetings)",
     "All standing committees held meetings during the period under review.",
     "Management should continue to ensure that committee meetings are effectively conducted and that all resolutions are implemented to strengthen governance and accountability.",
     "Management has developed annual meeting calendars, instructed secretaries to maintain accurate minutes, and introduced action trackers to monitor implementation of resolutions."),
    (2, "Budget Execution",
     "Total revenue collection stood at 82%, while total expenditure reached 84% of the approved budget. This reflects expenditure running slightly ahead of revenue collection, posing a risk of overruns.",
     "Expenditure should be closely aligned to actual revenue inflows, and budget performance should be reviewed regularly to avoid cash flow challenges.",
     "Management has instituted monthly budget reviews and adopted a cash flow management framework to match commitments with collections."),
    (3, "Income - Own Source Revenue",
     "Own source revenue collection stood at 78% of the budget, indicating underperformance due to low compliance in property rates and market levels.",
     "Revenue mobilization should be enhanced through enforcement, automation, and broadening the revenue base.",
     "Management has updated the valuation roll, intensified debt recovery, and introduced electronic payment platforms to improve compliance."),
    (4, "Accounting procedures and controls",
     "Segregation of duties was weak and bank reconciliations were delayed, increasing the risk of errors.",
     "Strengthen segregation of duties and ensure reconciliations are performed and reviewed promptly.",
     "Additional staff were assigned to accounts, reconciliations are now conducted monthly, and reviews are performed by senior officers."),
    (5, "Stores management",
     "Stock records were in place and consistently maintained, with access restricted to authorized personnel.",
     "Continue proper management of stores through regular stock verification and adherence to authorization procedures for issuing consumables.",
     "Monthly stock verifications are conducted, and stores access remains restricted to authorized personnel only."),
    (6, "Procurement procedures and controls",
     "All procurement activities fully complied with procurement regulations; however, some documentation was incomplete.",
     "Ensure strict adherence to the Public Procurement Act and maintain complete procurement records.",
     "Staff have undergone training in procurement compliance, and all procurement requests are now processed through the Electronic Government Procurement (EGPC) system."),
    (7, "Payroll management",
     "Payroll processes were up to date, and employee personal files were complete.",
     "Continue to ensure timely updates to payroll and maintain full employee records.",
     "The payroll has been cleaned, personal files updated, and quarterly payroll reviews introduced."),
    (8, "Risk management",
     "A risk register was in place; however, monitoring of mitigation strategies was limited.",
     "Continue maintaining the risk register and strengthen mechanisms to monitor risks regularly.",
     "Management has implemented a risk management framework, and departmental heads have been tasked with quarterly risk reporting."),
]

audit_df = pd.DataFrame([
    {"fiscal_year": YEAR, "period": PERIOD,
     "record_type": "audit_finding",
     "section": subject,
     "line_item": f"Finding {n}",
     "notes": f"FINDING: {finding}\nRECOMMENDATION: {rec}\nACTION TAKEN: {action}",
     "source_document": PDF_PATH.name,
     "source_page": 2}
    for n, subject, finding, rec, action in AUDIT_FINDINGS
])

print(f"✅ Audit findings: {len(audit_df)}")
audit_df[["section", "line_item"]]

✅ Audit findings: 8


,section,line_item
0,Governance (Established Committees & Meetings),Finding 1
1,Budget Execution,Finding 2
2,Income - Own Source Revenue,Finding 3
3,Accounting procedures and controls,Finding 4
4,Stores management,Finding 5
5,Procurement procedures and controls,Finding 6
6,Payroll management,Finding 7
7,Risk management,Finding 8


## 6. Debt & Arrears Position (Page 3)

The document reports outstanding amounts at two dates:
- **01-01-2025** (opening)
- **30-06-2025** (closing)

### Key finding
Total debt increased from **K18,241,433.33** to **K20,687,760.14** — a 
rise of **K2,446,326.81** in six months.

The largest components at 30 June 2025:
- ZRA PAYE: K7,342,242.12
- NAPSA Arrears (Penalties): K7,596,904.51
- PE's: K2,969,705.03
- NAPSA Arrears: K2,620,421.50

In [7]:
# ============================================================
# DEBT & ARREARS (verbatim from page 3)
# ============================================================
DEBT_ROWS = [
    ("PE's", 3091083.06, 2969705.03),
    ("ZAMTEL", 0, 0),
    ("ZESCO", 0, 0),
    ("WATER", 0, 0),
    ("LEGAL COSTS", 0, 0),
    ("NAPSA ARREARS", 2815853.18, 2620421.50),
    ("NAPSA ARREARS - PENALTIES", 7041602.00, 7596904.51),
    ("LASF ARREARS", -49973.08, -49973.08),
    ("LASF ARREARS - PENALTIES", 0, 0),
    ("OTHER PENSION ARREARS", 0, 0),
    ("OTHER PENSIONS ARREARS - PENALTIES", 0, 0),
    ("ZRA PAYE", 5186868.17, 7342242.12),
    ("ZRA - PENALTIES", 0, 0),
    ("GOODS & SERVICES", 156000.00, 208460.06),
]

debt_df = pd.DataFrame([
    {"fiscal_year": YEAR, "period": PERIOD,
     "record_type": "debt_arrears",
     "section": "Debt & Arrears",
     "line_item": item,
     "amount_prior_zmw": opening,
     "amount_current_zmw": closing,
     "source_document": PDF_PATH.name,
     "source_page": 3}
    for item, opening, closing in DEBT_ROWS
])

print(f"✅ Debt rows: {len(debt_df)}")
print(f"   Opening total (01-01-2025): K{debt_df['amount_prior_zmw'].sum():,.2f}")
print(f"   Closing total (30-06-2025): K{debt_df['amount_current_zmw'].sum():,.2f}")
print(f"   Change: K{debt_df['amount_current_zmw'].sum() - debt_df['amount_prior_zmw'].sum():,.2f}")

# Validate against document totals
assert abs(debt_df["amount_prior_zmw"].sum() - 18241433.33) < 1
assert abs(debt_df["amount_current_zmw"].sum() - 20687760.14) < 1
print("✅ Debt totals match published figures.")

✅ Debt rows: 14
   Opening total (01-01-2025): K18,241,433.33
   Closing total (30-06-2025): K20,687,760.14
   Change: K2,446,326.81
✅ Debt totals match published figures.


In [8]:
# ============================================================
# VALIDATION
# ============================================================
print("=" * 70)
print(f"VALIDATION — Chirundu Town Council {YEAR} {PERIOD}")
print("=" * 70)

checks = [
    ("Receipts budget total",
     receipts_df["approved_budget_zmw"].sum(), None),
    ("Receipts actual H1",
     receipts_df["actual_zmw"].sum(), None),
    ("Payments budget total",
     payments_df["approved_budget_zmw"].sum(), 103929415.85),
    ("Payments actual H1",
     payments_df["actual_zmw"].sum(), 43832931.75),
    ("Debt opening",
     debt_df["amount_prior_zmw"].sum(), 18241433.33),
    ("Debt closing",
     debt_df["amount_current_zmw"].sum(), 20687760.14),
]

for label, actual, expected in checks:
    if expected is None:
        print(f"   {label:<30} K{actual:>18,.2f}")
    else:
        diff = actual - expected
        status = "✅" if abs(diff) < 1 else "⚠️"
        print(f"{status} {label:<30} K{actual:>18,.2f}  (expected K{expected:,.2f})")

# Note on payments — document has 43,832,931.75 but our line items may differ
print(f"\n📌 If payments actual ≠ 43,832,931.75, one or more line items were")
print(f"   misread during OCR. Verify against the source PDF page 1.")

VALIDATION — Chirundu Town Council 2025 H1
   Receipts budget total          K    138,504,385.12
   Receipts actual H1             K     55,096,708.24
✅ Payments budget total          K    103,929,415.84  (expected K103,929,415.85)
✅ Payments actual H1             K     43,832,931.75  (expected K43,832,931.75)
✅ Debt opening                   K     18,241,433.33  (expected K18,241,433.33)
✅ Debt closing                   K     20,687,760.14  (expected K20,687,760.14)

📌 If payments actual ≠ 43,832,931.75, one or more line items were
   misread during OCR. Verify against the source PDF page 1.


In [9]:
# ============================================================
# EXPORT — SINGLE UNIFIED CSV (NO DELETION)
# ============================================================
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("finances_output")
OUTPUT_DIR.mkdir(exist_ok=True)

UNIVERSAL_COLS = [
    "fiscal_year", "period", "record_type", "section", "line_item",
    "amount_current_zmw", "amount_prior_zmw",
    "approved_budget_zmw", "actual_zmw",
    "variance_zmw", "variance_pct", "is_material_variance",
    "notes", "source_document", "source_page",
]


def normalize(df, record_type):
    out = pd.DataFrame()
    out["fiscal_year"] = df["fiscal_year"]
    out["period"] = df["period"]
    out["record_type"] = record_type
    out["section"] = df.get("section", "")
    out["line_item"] = df["line_item"]
    out["amount_current_zmw"] = df.get("amount_current_zmw")
    out["amount_prior_zmw"] = df.get("amount_prior_zmw")
    out["approved_budget_zmw"] = df.get("approved_budget_zmw")
    out["actual_zmw"] = df.get("actual_zmw")
    out["variance_zmw"] = df.get("variance_zmw")
    out["variance_pct"] = df.get("variance_pct")
    out["is_material_variance"] = (
        df["variance_pct"].abs() >= 20
        if "variance_pct" in df.columns else None
    )
    out["notes"] = df.get("notes", "")
    out["source_document"] = df["source_document"]
    out["source_page"] = df["source_page"]
    return out[UNIVERSAL_COLS]


receipts_norm = normalize(receipts_df, "receipt")
payments_norm = normalize(payments_df, "payment")
audit_norm = normalize(audit_df, "audit_finding")
debt_norm = normalize(debt_df, "debt_arrears")

combined_df = pd.concat(
    [receipts_norm, payments_norm, audit_norm, debt_norm],
    ignore_index=True,
)

out_file = OUTPUT_DIR / f"db-unza26-csc4792-chirundu_{YEAR}_financials.csv"
combined_df.to_csv(out_file, sep="|", index=False, encoding="utf-8-sig")

print(f"✅ Saved: {out_file.name}")
print(f"   Rows: {len(combined_df)}")
print(f"\nRows by record type:")
print(combined_df["record_type"].value_counts())

✅ Saved: db-unza26-csc4792-chirundu_2025_financials.csv
   Rows: 44

Rows by record type:
record_type
receipt          14
debt_arrears     14
payment           8
audit_finding     8
Name: count, dtype: int64


In [10]:
from pathlib import Path
OUTPUT_DIR = Path("finances_output")

print(f"📁 {OUTPUT_DIR.resolve()}")
print("=" * 70)
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        print(f"  📄 {f.name:<60} {f.stat().st_size/1024:>8.1f} KB")

📁 C:\Users\USER\Videos\chitundu\finances_output
  📄 chirundu_2023_raw_ocr.csv                                        60.0 KB
  📄 chirundu_2024_raw_ocr.csv                                        66.8 KB
  📄 chirundu_2025_raw_ocr.csv                                        25.9 KB
  📄 db-unza26-csc4792-chirundu_2022_financials.csv                   14.9 KB
  📄 db-unza26-csc4792-chirundu_2023_financials.csv                   16.0 KB
  📄 db-unza26-csc4792-chirundu_2024_financials.csv                   18.1 KB
  📄 db-unza26-csc4792-chirundu_2025_financials.csv                    8.4 KB
